# Multi-Model Performance Synthesizer

In [ ]:
import os
import asyncio
import tempfile
import subprocess
import time
import gradio as gr
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

In [ ]:
def benchmark_go_code(go_code: str, num_runs: int = 5) -> str:
    if go_code.startswith("// Error") or go_code.startswith("// Unexpected error"):
        return "Failed compilation/generation"

    env = os.environ.copy()
    env["PATH"] = "/usr/local/go/bin:" + env.get("PATH", "")

    with tempfile.TemporaryDirectory() as temp_dir:
        go_file_path = os.path.join(temp_dir, "main.go")
        binary_path = os.path.join(temp_dir, "main")

        with open(go_file_path, "w") as f:
            f.write(go_code)

        try:
            subprocess.run(
                ["go", "build", "-o", binary_path, go_file_path],
                check=True,
                capture_output=True,
                text=True,
                env=env
            )
        except FileNotFoundError:
            return "Compilation Failed: \"go\" compiler not found. Please install Go."
        except subprocess.CalledProcessError as e:
            return f"Compilation Failed: {e.stderr.strip()}"
        except Exception as e:
            return f"Compilation Failed: {e}"

        total_time = 0.0
        success_runs = 0

        for _ in range(num_runs):
            try:
                start_time = time.perf_counter()
                subprocess.run(
                    [binary_path],
                    check=True,
                    capture_output=True,
                    text=True
                )
                end_time = time.perf_counter()
                total_time += (end_time - start_time)
                success_runs += 1
            except FileNotFoundError:
                return "Execution Failed: Binary not found after compilation."
            except subprocess.CalledProcessError as e:
                return f"Execution Failed: {e.stderr.strip()}"
            except Exception as e:
                return f"Execution Failed: {e}"

        if success_runs == 0:
            return "Execution Failed"

        avg_time = total_time / success_runs
        return f"{avg_time:.6f}s (avg over {success_runs} runs)"

## 2. Model Orchestration (Phases 1 & 2)

In [ ]:
SYSTEM_PROMPT = """You are an expert Go developer. Translate the provided Python code into highly optimized Go code. 
The Go code must print the final result to stdout. Do not include markdown formatting, explanations, or backticks in your response. 
Return ONLY raw, valid, compilable Go code starting with `package main`."""

openai_client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

openrouter_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

async def generate_single_model(python_code: str, model_name: str, client: AsyncOpenAI) -> str:
    try:
        response = await client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": python_code}
            ],
            temperature=0.2
        )
        content = response.choices[0].message.content
        if content.startswith("```go"):
            content = content[5:]
        elif content.startswith("```"):
            content = content[3:]
        if content.endswith("```"):
            content = content[:-3]
        return content.strip()
    except Exception as e:
        return f"// Error generating code: {e}"

async def generate_all_models(python_code: str):
    t1 = generate_single_model(python_code, "gpt-4o", openai_client)
    t2 = generate_single_model(python_code, "qwen/qwen-2.5-coder-32b-instruct", openrouter_client)
    t3 = generate_single_model(python_code, "deepseek-coder:6.7b", ollama_client)
    
    results = await asyncio.gather(t1, t2, t3, return_exceptions=True)
    
    safe_results = []
    for r in results:
        if isinstance(r, Exception):
            safe_results.append(f"// Unexpected error: {str(r)}")
        else:
            safe_results.append(str(r))
            
    return safe_results[0], safe_results[1], safe_results[2]

## 3. UI and Leaderboard (Phase 4)

In [ ]:
async def handle_synthesize(python_code: str):
    openai_go, openrouter_go, ollama_go = await generate_all_models(python_code)
    
    openai_bench = benchmark_go_code(openai_go)
    openrouter_bench = benchmark_go_code(openrouter_go)
    ollama_bench = benchmark_go_code(ollama_go)
    
    def parse_time(result_str):
        try:
            if "avg" in result_str:
                return float(result_str.split("s")[0])
            return float('inf')
        except:
            return float('inf')

    results = [
        ("OpenAI (gpt-4o)", openai_bench),
        ("OpenRouter (Qwen)", openrouter_bench),
        ("Ollama (DeepSeek)", ollama_bench)
    ]
    results.sort(key=lambda x: parse_time(x[1]))
    
    leaderboard = "### Leaderboard\n"
    for idx, (model, bench) in enumerate(results, 1):
        leaderboard += f"{idx}. {model}: {bench}\n"
        
    return openai_go, openrouter_go, ollama_go, leaderboard

with gr.Blocks(title="Multi-Model Performance Synthesizer") as ui:
    gr.Markdown("# Multi-Model Performance Synthesizer")
    
    with gr.Row():
        python_input = gr.Code(label="Python Input", language="python", lines=15)
        
    synthesize_btn = gr.Button("Synthesize & Benchmark", variant="primary")
    
    with gr.Row():
        openai_output = gr.Code(label="OpenAI (GPT) Go Code", language="c", lines=15)
        openrouter_output = gr.Code(label="OpenRouter (Qwen) Go Code", language="c", lines=15)
        ollama_output = gr.Code(label="Ollama (DeepSeek) Go Code", language="c", lines=15)

    with gr.Row():
        leaderboard_output = gr.Markdown("### Leaderboard \n (Run benchmarks to see results)")

    synthesize_btn.click(
        fn=handle_synthesize,
        inputs=[python_input],
        outputs=[openai_output, openrouter_output, ollama_output, leaderboard_output]
    )

In [ ]:
ui.launch(inbrowser=True)